# FPL Weekly Assistant

Every week: update **FREE_TRANSFERS**, add any injury news you know that the model
can't read, then run the report.

The engine lives in one file that this notebook and the backtest both import, so
the settings you tune in the backtest are the settings that run here.


In [1]:
# Setup - fetches the engine. Run once per session.
import os, sys, subprocess

# The one copy of the engine. Point this at your repo once you've pushed it;
# if fpl_engine.py is already sitting next to this notebook it is used as-is,
# so local edits work without pushing anything.
ENGINE_URL = "https://raw.githubusercontent.com/PKnott/fpl/main/fpl_engine.py"

if not os.path.exists("fpl_engine.py"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pulp", "requests"])
    import requests
    r = requests.get(ENGINE_URL, timeout=60); r.raise_for_status()
    open("fpl_engine.py", "w").write(r.text)
    print("engine downloaded")

import fpl_engine as E
print("engine ready")


engine ready


## Your settings

Edit these. Everything else - the model, the planner, the chip rules - lives in
`E.DEFAULTS` and is what the backtest tunes; you shouldn't need to touch it.


In [2]:
E.USER.update(
    TEAM_ID=7399216,        # the number in your "Points" page URL
    FREE_TRANSFERS=1,       # the one thing FPL doesn't publish - update weekly

    # Injury news the model can't read: {player: {gameweek: chance 0-1}}
    AVAILABILITY_OVERRIDES={},
    # How often you expect a player to start (0-1), e.g. a new signing:
    START_OVERRIDES={},
    LOCK=[],                # never sell
    EXCLUDE=[],             # never buy
)

cfg = E.resolve_cfg()
print(f"planning {cfg['PLAN_WEEKS']} weeks, long view {cfg['LONG_VIEW']}, "
      f"chip thresholds {cfg['CHIP_TOP_PCT']}")


planning 5 weeks, long view 19, chip thresholds {'3xc': 0.15, 'bboost': 0.15, 'freehit': 0.15, 'wildcard': 0.15}


## The weekly report

In [3]:
report = E.run(cfg)

Loading player histories for 182 players...
 FPL ASSISTANT - GW6   plan GW6-10, long view to GW24, projections to GW29
Bank £2.0m | Free transfers 1 | Chips: Bench Boost, Triple Captain

YOUR SQUAD
Player                    Sell  Next   12wk  Fixture / price outlook
GK Verbruggen              4.5   3.2   16.5  SUN (A)
GK Woodman                 4.0   0.0    0.0  MCI (H)
DEF Calafiori              5.7   5.2   23.9  LEE (H) ↑ likely rise
DEF Gvardiol               5.7   3.7   22.4  LIV (A)
DEF De Cuyper              4.7   4.1   22.4  SUN (A) ↑ likely rise
DEF Konsa                  4.5   3.8   17.7  LEE (H)
DEF Mendy                  4.0   1.3    6.3  EVE (H)
MID Tavernier              6.1   5.0   24.0  CHE (A)
MID Groß                   5.6   4.5   23.7  SUN (A)
MID Ødegaard               6.7   4.6   21.1  LEE (H)
MID Rogers                 7.6   4.3   20.8  BOU (H)
MID Gakpo                  7.2   3.6   18.3  MCI (H)
FWD Haaland               15.5   5.4   33.4  LIV (A)
FWD Isak        

## Why is a player projected what he is?

Add any names - yours or not. Ambiguous names take a club: `"Gakpo|LIV"`.


In [4]:
report.explain("Haaland", "Saka")


POINTS BREAKDOWN - GW6
                                   Haaland            Saka
Fixture                            LIV (A)         LEE (H)
Starts this season                  5 of 5          5 of 5
Last 5 games (newest 1st)   90,90,90,90,90  80,90,89,90,67
Chance of starting                    100%            100%
Chance of 60+ minutes                 100%            100%
Expected minutes                        90              84
Availability                          100%            100%
----------------------------------------------------------
Playing time                         +2.00           +2.00
Goals                                +2.10           +2.22
Assists                              +0.19           +0.72
Clean sheet                          +0.00           +0.50
Def. contributions                   +0.00           +0.07
Bonus                                +1.17           +0.72
Yellow cards                         -0.04           -0.05
--------------------------------

## Team defence: the rating next to what actually happened

In [5]:
report.team_table()


TEAM DEFENCE (league avg goals per team per game: 1.51)
Team   Games  Conceded  xG agst  Pre-season  Keeper busy  Adjust   RATING
              per game per game      rating     shots/90
IPS        5      2.20     1.80        115%          6.2    1.19     141%
CRY        5      2.20     1.89        100%          6.7    1.23     136%
NEW        5      1.80     1.83        108%          5.6    1.13     127%
CHE        5      2.40     1.78         85%          7.0    1.26     126%
FUL        5      1.60     1.82        108%          4.8    1.04     117%
AVL        5      1.80     1.84         92%          5.0    1.07     110%
COV        5      2.00     1.73        115%          3.8    0.93     107%
HUL        5      0.80     1.48        115%          4.2    0.98     103%
LEE        5      0.60     1.29        108%          4.6    1.02      98%
TOT        5      1.60     1.56        100%          4.0    0.95      96%
SUN        5      2.00     1.73        108%          3.2    0.85      95

## Chip detail

Every remaining week of each chip's window, so you can see what the ranking is
built on rather than taking the verdict on trust.


In [6]:
import pandas as pd
rows = {E.CHIP_NAMES[c]: v["values"] for c, v in report.chips.items()}
if rows:
    display(pd.DataFrame(rows).rename_axis("GW").round(1))
else:
    print("No chips available.")


,Bench Boost,Triple Captain
GW,,
6,9.4,5.7
7,8.4,9.1
8,10.5,6.8
9,13.0,7.0
10,15.4,6.1
11,15.4,8.0
12,14.3,6.0
13,15.5,7.2
14,14.7,6.1
